# v6b MIXED baseline (GPU 0): all three languages at once, never switching
A new model from scratch, trained on Hindi, Chinese and Arabic **at the same time**: inside every phase the batches alternate Hindi, Chinese, Arabic, Hindi, ... (each batch is one language, so the script-restricted loss is unchanged). Same model, schedule and evaluation as the continual and Hindi-only runs.
Question: does a model that knows all three languages, but never switches, form the same kind of self?
**Just Run All.** Output appears below. Rerunning resumes from the last saved phase. Checkpoints: `runs/v6b_mixed_s0/ckpt/pXXX_mix.pt`.

In [ ]:
import os, sys, glob, json, time
os.environ["CUDA_VISIBLE_DEVICES"] = "0"     # run this cell FIRST in a fresh kernel
os.environ["HF_HUB_DISABLE_XET"] = "1"          # HF uploads failed overnight in the new "xet" path; use the classic upload
REPO = "/workspace/shreshth"
os.chdir(REPO); sys.path.insert(0, REPO)
from common_v6 import *
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")
RUN = "v6b_mixed_s0"

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
for attempt in range(5):                          # retry: HF sometimes drops the connection
    try:
        api = HfApi(); HF_REPO = api.whoami()["name"] + "/cml-lang-self-checkpoints"; break
    except Exception as e:
        print("HF retry", attempt, type(e).__name__); time.sleep(10)

# FLORES dev + devtest
from datasets import load_dataset
for l, code in {"en": "eng_Latn", "hi": "hin_Deva", "zh": "cmn_Hans", "ar": "arb_Arab"}.items():
    for split in ["dev", "devtest"]:
        p = f"data/flores/{split}.{l}.jsonl"
        if not os.path.exists(p):
            os.makedirs("data/flores", exist_ok=True)
            ds = load_dataset("openlanguagedata/flores_plus", code, split=split)
            with open(p, "w", encoding="utf-8") as f:
                for t in ds["text"]:
                    f.write(json.dumps({"text": t}, ensure_ascii=False) + "\n")
missing = [p for p in [f"data/en-{l}/{s}.jsonl" for l in LANGS for s in ["train", "val"]] if not os.path.exists(p)]
print("data OK" if not missing else f"MISSING: {missing}")

In [ ]:
C = RunConfig(run_name=RUN, mode="mixed", n_phases=60, seed=0, time_cap_h=120.0, hf_repo=HF_REPO)
print(C)

In [ ]:
D = load_data(train_langs=["hi", "zh", "ar"])

In [ ]:
model = train_run(C, D)

In [ ]:
print_run_summary(RUN)